In [18]:
import pandas as pd
import numpy as np

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [20]:
df = pd.read_csv("appeal_dataset.csv")

print("Shape:", df.shape)
df.head()

Shape: (5000, 14)


,appeal_id,patient_age,procedure,denial_reason,medical_necessity_score,documentation_completeness_pct,patient_severity,previous_treatment_failed,clinical_guideline_match,previous_authorization_history,appeal_submitted,appeal_success_probability,actual_appeal_outcome,predicted_appeal_outcome
0,A00001,69,MRI Spine,Insufficient Documentation,47,78,Medium,Yes,Yes,Previously Approved,Yes,0.633,Rejected,Approved
1,A00002,32,Genetic Testing,Eligibility Criteria,38,59,Low,No,No,No Prior Request,Yes,0.080,Rejected,Rejected
2,A00003,78,Genetic Testing,Insufficient Documentation,46,70,High,No,Yes,Previously Approved,Yes,0.527,Rejected,Approved
3,A00004,38,Physical Therapy,Not Medically Necessary,67,32,Medium,Yes,Yes,Previously Approved,Yes,0.412,Approved,Rejected
4,A00005,41,Knee Surgery,Eligibility Criteria,38,69,Medium,No,Yes,No Prior Request,Yes,0.207,Rejected,Rejected


In [21]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   appeal_id                       5000 non-null   str    
 1   patient_age                     5000 non-null   int64  
 2   procedure                       5000 non-null   str    
 3   denial_reason                   5000 non-null   str    
 4   medical_necessity_score         5000 non-null   int64  
 5   documentation_completeness_pct  5000 non-null   int64  
 6   patient_severity                5000 non-null   str    
 7   previous_treatment_failed       5000 non-null   str    
 8   clinical_guideline_match        5000 non-null   str    
 9   previous_authorization_history  5000 non-null   str    
 10  appeal_submitted                5000 non-null   str    
 11  appeal_success_probability      5000 non-null   float64
 12  actual_appeal_outcome           5000 non-null

In [22]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [23]:
df = df.drop([
    "appeal_id",
    "appeal_submitted",
    "appeal_success_probability",
    "predicted_appeal_outcome"
], axis=1)

print(df.columns)

Index(['patient_age', 'procedure', 'denial_reason', 'medical_necessity_score',
       'documentation_completeness_pct', 'patient_severity',
       'previous_treatment_failed', 'clinical_guideline_match',
       'previous_authorization_history', 'actual_appeal_outcome'],
      dtype='str')


In [24]:
X = df.drop("actual_appeal_outcome", axis=1)

y = df["actual_appeal_outcome"]

In [25]:
y = y.map({
    "Rejected": 0,
    "Approved": 1
})

print(y.value_counts())

actual_appeal_outcome
0    3048
1    1952
Name: count, dtype: int64


In [26]:
numerical_columns = [
    "patient_age",
    "medical_necessity_score",
    "documentation_completeness_pct"
]

categorical_columns = [
    "procedure",
    "denial_reason",
    "patient_severity",
    "previous_treatment_failed",
    "clinical_guideline_match",
    "previous_authorization_history"
]

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (4000, 9)
Testing: (1000, 9)


In [28]:
binary_columns = [
    "previous_treatment_failed",
    "clinical_guideline_match"
]

for column in binary_columns:
    X_train[column] = X_train[column].map({
        "No": 0,
        "Yes": 1
    })

    X_test[column] = X_test[column].map({
        "No": 0,
        "Yes": 1
    })

In [29]:
severity_mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

X_train["patient_severity"] = X_train["patient_severity"].map(
    severity_mapping
)

X_test["patient_severity"] = X_test["patient_severity"].map(
    severity_mapping
)

In [ ]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_columns = ['procedure', 'denial_reason', 'previous_authorization_history']

X_train_ohe = ohe.fit_transform(X_train[ohe_columns])
X_test_ohe = ohe.transform(X_test[ohe_columns])

ohe_feature_names = ohe.get_feature_names_out(ohe_columns)
X_train_ohe_df = pd.DataFrame(X_train_ohe, columns=ohe_feature_names, index=X_train.index)
X_test_ohe_df = pd.DataFrame(X_test_ohe, columns=ohe_feature_names, index=X_test.index)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train[numerical_columns])
X_test_scaled = scaler.transform(X_test[numerical_columns])

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=numerical_columns, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=numerical_columns, index=X_test.index)

In [ ]:
remaining_columns = ['previous_treatment_failed', 'clinical_guideline_match', 'patient_severity']

X_train_preprocessed = pd.concat([X_train_scaled_df, X_train[remaining_columns], X_train_ohe_df], axis=1)
X_test_preprocessed = pd.concat([X_test_scaled_df, X_test[remaining_columns], X_test_ohe_df], axis=1)

print('Preprocessed Train shape:', X_train_preprocessed.shape)
print('Preprocessed Test shape:', X_test_preprocessed.shape)

In [ ]:
train_preprocessed = X_train_preprocessed.copy()
train_preprocessed['actual_appeal_outcome'] = y_train

test_preprocessed = X_test_preprocessed.copy()
test_preprocessed['actual_appeal_outcome'] = y_test

print('Train columns count:', len(train_preprocessed.columns))

In [ ]:
train_preprocessed.to_csv('train_preprocessed.csv', index=False)
test_preprocessed.to_csv('test_preprocessed.csv', index=False)
print('Saved train_preprocessed.csv and test_preprocessed.csv successfully.')

In [ ]:
train_preprocessed.head()